# Week 15: Agent Studio I: a bounded analysis workflow

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Define a structured measurement request

This deterministic workflow is a baseline for an engineering assistant. It accepts numeric readings and returns summary evidence without inventing a physical diagnosis.


In [ ]:
request={"task":"summarize","units":"synthetic units","readings":[3.1,3.2,3.0,3.6,3.4]}
def summarize(request):
    values=request.get("readings")
    if request.get("task")!="summarize" or not isinstance(values,list) or len(values)<2:
        raise ValueError("Need a summarize request with at least two readings")
    if not all(type(x) in (int,float) and np.isfinite(x) for x in values): raise ValueError("Need finite numbers")
    return {"n":len(values),"mean":float(np.mean(values)),"sample_std":float(np.std(values,ddof=1)),"units":request["units"]}
result=summarize(request); print(result)


**Checkpoint.** The sample standard deviation uses ddof=1. Units remain part of the result record.


## 3. Trace every operation

A reproducible workflow can be independently rerun. Preserve the exact input and a deterministic digest.


In [ ]:
payload=json.dumps(request,sort_keys=True).encode()
trace={"request_sha256":hashlib.sha256(payload).hexdigest(),"request":request,"result":result,
       "limitation":"descriptive statistics; no calibrated diagnostic threshold"}
assert summarize(request)==result
print(trace["request_sha256"])


**Checkpoint.** The hash identifies the request; it does not certify measurement authenticity.


## 4. Test a malformed request

A failed input check is a defined behavior. Record the error rather than treating it as a model hallucination.


In [ ]:
try:
    summarize({"task":"summarize","units":"u","readings":["unknown",2]})
except ValueError as error:
    failure=str(error)
else:
    raise AssertionError("Invalid request should fail")
summary={"trace":trace,"invalid_input_error":failure}
print(summary["invalid_input_error"])


**Checkpoint.** A later LLM interface should produce this same schema and must still pass the validator.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":15,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-15-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)
